# AKI ROC Analysis
**Paper:** *Predictive Value of Hydration Markers for AKI in Young vs. Older Females*

In [ ]:
pkgs <- c("pROC", "ggplot2", "dplyr", "zoo", "coin", "rstatix")
install.packages(pkgs[!pkgs %in% installed.packages()[,"Package"]])

In [ ]:
library(pROC); library(ggplot2); library(dplyr); library(coin); library(rstatix)
cat("Libraries loaded.\n")

In [ ]:
# Set working directory and load
setwd("C:/Users/jwh18b/OneDrive - Florida State University/MANUSCRIPTS/Write_Kidney/aki_stats")
cat("Working directory:", getwd(), "\n")

df_raw <- read.csv("AKI_data.csv", na.strings = c("", "NA", "NaN"))
colnames(df_raw)[1] <- "Age_Group"

cat("Columns:\n"); print(colnames(df_raw))
cat("Age_Group unique values:", paste(unique(df_raw$Age_Group), collapse=", "), "\n")

# Keep full copy for Table 1 (includes AH13 who has no AKI Risk)
df_demo <- df_raw

# ROC dataset: drop rows with missing AKI Risk
df_raw        <- df_raw[!is.na(df_raw$AKI.Risk), ]
df_raw$AKI_binary <- ifelse(tolower(df_raw$AKI.Risk) == "yes", 1, 0)

cat("\nRows after dropping missing AKI Risk:", nrow(df_raw), "\n")
print(table(df_raw$Age_Group, df_raw$AKI_binary,
            dnn = c("Group", "AKI (0=no,1=yes)")))

In [ ]:
# Marker definitions and missing value check
markers       <- c("Spot.USG","X24hr.USG","X24hr.Osmo","Plasma.Osmo","percent.change.weight")
marker_labels <- c("Spot USG","24-hr USG","24-hr Osmo","Plasma Osmo","% Body Mass Change")

cat("=== Missing values ===\n")
for (i in seq_along(markers))
  cat(sprintf("  %-22s  missing=%d  available=%d\n",
              marker_labels[i],
              sum(is.na(df_raw[[markers[i]]])),
              sum(!is.na(df_raw[[markers[i]]]))))



In [ ]:
# Table 1 — uses df_demo so all 26 participants are included
demo       <- df_demo[!duplicated(df_demo$ID), ]
young_demo <- demo[demo$Age_Group == "Young", ]
older_demo <- demo[demo$Age_Group == "Older", ]
cat(sprintf("Participants: YF n=%d  |  OF n=%d\n\n", nrow(young_demo), nrow(older_demo)))

table1_var <- function(var_name, label, y_df, o_df) {
  y <- na.omit(y_df[[var_name]])
  o <- na.omit(o_df[[var_name]])
  wt  <- wilcox.test(y, o, exact=FALSE)
  rbc <- 1 - (2 * as.numeric(wt$statistic)) / (length(y) * length(o))
  cat(sprintf("%s:\n  YF: %.1f [%.1f]   OF: %.1f [%.1f]   p=%.4f   rbc=%.2f\n\n",
              label, median(y), IQR(y), median(o), IQR(o), wt$p.value, rbc))
}

cat("=== TABLE 1 ===\n\n")
table1_var("Age",            "Age (years)", young_demo, older_demo)
table1_var("Screening.BMI", "BMI (kg/m2)", young_demo, older_demo)

In [ ]:
# Helpers
safe_roc <- function(response, predictor, direction="auto") {
  ok <- complete.cases(response, predictor)
  if (sum(ok) < 5 || length(unique(response[ok])) < 2) return(NULL)
  tryCatch(roc(response[ok], predictor[ok], quiet=TRUE, direction=direction),
           error=function(e) NULL)
}

roc_summary <- function(roc_obj) {
  if (is.null(roc_obj)) return(NULL)
  auc_val <- as.numeric(auc(roc_obj))
  ci_obj  <- tryCatch(ci.auc(roc_obj, conf.level=0.95), error=function(e) NULL)
  sens <- rev(roc_obj$sensitivities)
  spec <- rev(roc_obj$specificities)
  thre <- rev(roc_obj$thresholds)
  J    <- sens + spec - 1
  best <- which.max(J)
  if (length(best) == 0) best <- 1L
  list(auc       = auc_val,
       ci_lo     = if (!is.null(ci_obj)) as.numeric(ci_obj[1]) else NA_real_,
       ci_hi     = if (!is.null(ci_obj)) as.numeric(ci_obj[3]) else NA_real_,
       J         = J[best],
       threshold = thre[best],
       sens      = sens[best],
       spec      = spec[best],
       n         = length(roc_obj$cases) + length(roc_obj$controls))
}

s1 <- function(x) if (length(x) == 1 && !is.null(x)) x else NA_real_

delong_unpaired <- function(r1, r2) {
  if (is.null(r1)||is.null(r2)) return(list(p=NA_real_,z=NA_real_))
  tryCatch({ t <- roc.test(r1,r2,method="delong",paired=FALSE)
             list(p=t$p.value, z=as.numeric(t$statistic)) },
           error=function(e) list(p=NA_real_,z=NA_real_))
}

delong_paired <- function(r1, r2) {
  if (is.null(r1)||is.null(r2)) return(list(p=NA_real_,z=NA_real_))
  tryCatch({ t <- roc.test(r1,r2,method="delong",paired=TRUE)
             list(p=t$p.value, z=as.numeric(t$statistic)) },
           error=function(e) list(p=NA_real_,z=NA_real_))
}


# Determine ROC direction for each marker using the full dataset
# This prevents direction flipping between age groups
marker_directions <- sapply(markers, function(m) {
  r <- safe_roc(df_raw$AKI_binary, df_raw[[m]])
  if (is.null(r)) return("auto") else return(r$direction)
})
cat("ROC directions:\n"); print(marker_directions)

cat("Helpers and directions ready.\n")


In [ ]:
# ── Cell 7: Aim 1 ─────────────────────────────────────────────────────────────
young <- df_raw[df_raw$Age_Group == "Young", ]
older <- df_raw[df_raw$Age_Group == "Older", ]
cat(sprintf("Observations — Young: %d  |  Older: %d\n\n", nrow(young), nrow(older)))

aim1_results <- data.frame()

for (i in seq_along(markers)) {
  m   <- markers[i]
  lab <- marker_labels[i]

  # direction = "auto": each subgroup independently chooses the direction
  # that maximises AUC (consistent with jamovi behaviour)
  ry <- safe_roc(young$AKI_binary, young[[m]])
  ro <- safe_roc(older$AKI_binary, older[[m]])

  sy <- roc_summary(ry)
  so <- roc_summary(ro)
  dl <- delong_unpaired(ry, ro)

  cat(sprintf("--- %s ---\n", lab))
  if (!is.null(sy))
    cat(sprintf("  YF (n = %d): AUC = %.2f (95%% CI %.2f\u2013%.2f)  J = %.3f  threshold > %.4f  Sens = %.2f  Spec = %.2f\n",
                sy$n, sy$auc, sy$ci_lo, sy$ci_hi, sy$J, sy$threshold, sy$sens, sy$spec))
  if (!is.null(so))
    cat(sprintf("  OF (n = %d): AUC = %.2f (95%% CI %.2f\u2013%.2f)  J = %.3f  threshold > %.4f  Sens = %.2f  Spec = %.2f\n",
                so$n, so$auc, so$ci_lo, so$ci_hi, so$J, so$threshold, so$sens, so$spec))
  cat(sprintf("  DeLong (unpaired): Z = %.3f  p = %.4f\n\n",
              ifelse(is.na(dl$z), 0, dl$z),
              ifelse(is.na(dl$p), 1, dl$p)))

  aim1_results <- bind_rows(aim1_results,
    data.frame(marker = lab, group = "YF",
               auc = s1(sy$auc), ci_lo = s1(sy$ci_lo), ci_hi = s1(sy$ci_hi),
               J = s1(sy$J), threshold = s1(sy$threshold),
               sens = s1(sy$sens), spec = s1(sy$spec), delong_p = s1(dl$p)),
    data.frame(marker = lab, group = "OF",
               auc = s1(so$auc), ci_lo = s1(so$ci_lo), ci_hi = s1(so$ci_hi),
               J = s1(so$J), threshold = s1(so$threshold),
               sens = s1(so$sens), spec = s1(so$spec), delong_p = s1(dl$p))
  )
}

In [ ]:
# Aim 2 — Paired DeLong: Spot vs 24-hr USG
ok      <- complete.cases(df_raw$Spot.USG, df_raw$X24hr.USG, df_raw$AKI_binary)
df_c    <- df_raw[ok, ]
cat(sprintf("Complete cases for paired test: %d\n\n", nrow(df_c)))

roc_spot <- safe_roc(df_c$AKI_binary, df_c$Spot.USG)
roc_24hr <- safe_roc(df_c$AKI_binary, df_c$X24hr.USG)
s_spot   <- roc_summary(roc_spot)
s_24hr   <- roc_summary(roc_24hr)
dl_aim2  <- delong_paired(roc_spot, roc_24hr)

cat("Spot USG:\n")
cat(sprintf("  AUC=%.2f (95%% CI %.2f-%.2f)  J=%.3f  thr>%.4f  Sens=%.2f  Spec=%.2f\n",
            s_spot$auc,s_spot$ci_lo,s_spot$ci_hi,s_spot$J,s_spot$threshold,s_spot$sens,s_spot$spec))
cat("\n24-hr USG:\n")
cat(sprintf("  AUC=%.2f (95%% CI %.2f-%.2f)  J=%.3f  thr>%.4f  Sens=%.2f  Spec=%.2f\n",
            s_24hr$auc,s_24hr$ci_lo,s_24hr$ci_hi,s_24hr$J,s_24hr$threshold,s_24hr$sens,s_24hr$spec))
cat(sprintf("\nPaired DeLong: Z=%.3f  p=%.4f\n", dl_aim2$z, dl_aim2$p))

In [ ]:
# Figure 1 — AUC dot plot
options(repr.plot.width=11, repr.plot.height=7)
COL_YOUNG <- "#5CB8B2"; COL_OLDER <- "#FFC72C"; COL_TEXT <- "#1a1a1a"

dot_data        <- aim1_results
dot_data$marker <- factor(dot_data$marker, levels=rev(marker_labels))
dot_data$group  <- factor(dot_data$group,  levels=c("YF","OF"))

annot <- dot_data %>%
  group_by(marker) %>%
  summarise(delong_p=first(delong_p), .groups="drop") %>%
  mutate(p_label=ifelse(delong_p < 0.05,
                        sprintf("p=%.3f *",delong_p),
                        sprintf("p=%.2f",  delong_p)))

theme_dot <- theme_classic(base_size=18) +
  theme(text=element_text(color=COL_TEXT),
        axis.title=element_text(size=20),
        axis.text=element_text(size=16,color=COL_TEXT),
        axis.line=element_line(linewidth=1.5,color=COL_TEXT),
        axis.ticks=element_line(linewidth=1.2,color=COL_TEXT),
        legend.position="top", legend.title=element_blank(),
        legend.text=element_text(size=16),
        plot.title=element_text(size=18,face="bold",hjust=0.5),
        panel.grid.major.x=element_line(color="grey90",linewidth=0.8))

p_dot <- ggplot(dot_data, aes(x=auc, y=marker, color=group)) +
  geom_vline(xintercept=0.5, linetype="dashed", color="grey60", linewidth=1.0) +
  geom_errorbar(aes(xmin=ci_lo, xmax=ci_hi), width=0.2, linewidth=1.0,
                position=position_dodge(width=0.5), orientation="y") +
  geom_point(size=4.5, position=position_dodge(width=0.5)) +
  geom_text(data=annot, aes(x=1.02,y=marker,label=p_label),
            color="grey30",size=4.5,hjust=0,inherit.aes=FALSE) +
  scale_color_manual(values=c(YF=COL_YOUNG,OF=COL_OLDER),
                     labels=c(YF="Young females",OF="Older females")) +
  scale_x_continuous(name="Area Under the Curve (AUC)", limits=c(0.0,1.18),
                     breaks=c(0.25,0.5,0.75,1.0),
                     labels=c("0.25","0.50","0.75","1.00")) +
  scale_y_discrete(name=NULL) +
  labs(title="AUC by Hydration Marker and Age Group") + theme_dot

dir.create("figures", showWarnings=FALSE)
ggsave("figures/Fig1_AUC_dotplot.png", p_dot, width=11, height=7, dpi=300, bg="white")
cat("Saved: figures/Fig1_AUC_dotplot.png\n")
p_dot

In [ ]:
# Figure 2 — Paired ROC curve
options(repr.plot.width=9, repr.plot.height=8)
COL_SPOT <- "#A6192E"; COL_24HR <- "#FFC72C"; COL_CI <- "#aaaaaa"; COL_DIAG <- "grey55"

roc_to_df <- function(roc_obj, label) {
  if (is.null(roc_obj)) return(NULL)
  data.frame(fpr=1-rev(roc_obj$specificities), tpr=rev(roc_obj$sensitivities), label=label)
}

ci_band <- function(roc_obj, n_boot=200) {
  if (is.null(roc_obj)) return(NULL)
  ci_obj <- tryCatch(
    ci.se(roc_obj, specificities=seq(0,1,0.01), conf.level=0.95,
          method="bootstrap", boot.n=n_boot, progress="none"),
    error=function(e) NULL)
  if (is.null(ci_obj)) return(NULL)
  data.frame(fpr=1-as.numeric(rownames(ci_obj)),
             lower=as.numeric(ci_obj[,1]), upper=as.numeric(ci_obj[,3]))
}

p_fmt <- function(p) {
  if (is.na(p)) return("p = NA")
  if (p < 0.001) return("p < 0.001")
  sprintf("p = %.3f", p)
}

theme_roc <- theme_classic(base_size=20) +
  theme(text=element_text(color=COL_TEXT),
        axis.title=element_text(size=22),
        axis.text=element_text(size=18,color=COL_TEXT),
        axis.line=element_line(linewidth=1.8,color=COL_TEXT),
        axis.ticks=element_line(linewidth=1.4,color=COL_TEXT),
        plot.title=element_text(size=20,face="bold",hjust=0.5),
        legend.position="none")

ci_spot_band <- ci_band(roc_spot)
ci_24hr_band <- ci_band(roc_24hr)
df_spot      <- roc_to_df(roc_spot, "spot")
df_24        <- roc_to_df(roc_24hr, "24hr")
ci_spot_auc  <- ci.auc(roc_spot)
ci_24hr_auc  <- ci.auc(roc_24hr)

label_spot <- sprintf("Spot USG\nAUC = %.2f (%.2f-%.2f)",
                      as.numeric(auc(roc_spot)),
                      as.numeric(ci_spot_auc[1]), as.numeric(ci_spot_auc[3]))
label_24hr <- sprintf("24-hr USG\nAUC = %.2f (%.2f-%.2f)",
                      as.numeric(auc(roc_24hr)),
                      as.numeric(ci_24hr_auc[1]), as.numeric(ci_24hr_auc[3]))

p_roc <- ggplot(df_24, aes(x=fpr, y=tpr)) +
  { if (!is.null(ci_24hr_band))
      geom_ribbon(data=ci_24hr_band, aes(x=fpr,ymin=lower,ymax=upper),
                  fill=COL_CI, alpha=0.18, inherit.aes=FALSE) } +
  { if (!is.null(ci_spot_band))
      geom_ribbon(data=ci_spot_band, aes(x=fpr,ymin=lower,ymax=upper),
                  fill=COL_CI, alpha=0.18, inherit.aes=FALSE) } +
  geom_abline(slope=1,intercept=0,linetype="dashed",color=COL_DIAG,linewidth=1.0) +
  geom_step(data=df_24,   aes(x=fpr,y=tpr), color=COL_24HR, linewidth=2.0) +
  geom_step(data=df_spot, aes(x=fpr,y=tpr), color=COL_SPOT,  linewidth=2.0) +
  annotate("text",x=0.56,y=0.30,label=label_24hr,
           color=COL_24HR,size=5.5,hjust=0,fontface="bold",lineheight=1.25) +
  annotate("text",x=0.56,y=0.14,label=label_spot,
           color=COL_SPOT, size=5.5,hjust=0,fontface="bold",lineheight=1.25) +
  annotate("text",x=0.98,y=0.07,label=paste("DeLong",p_fmt(dl_aim2$p)),
           color=COL_TEXT,size=5.2,hjust=1,fontface="italic") +
  annotate("text",x=0.98,y=0.01,label=paste0("n = ",nrow(df_c)),
           color="grey50",size=4.8,hjust=1) +
  scale_x_continuous(name="False Positive Rate",limits=c(0,1),
                     breaks=c(0,.25,.5,.75,1),expand=expansion(mult=c(0.01,0.02))) +
  scale_y_continuous(name="Sensitivity",limits=c(0,1),
                     breaks=c(0,.25,.5,.75,1),expand=expansion(mult=c(0.01,0.02))) +
  labs(title="ROC: Spot vs 24-hr USG") +
  coord_cartesian(clip="off") + theme_roc

ggsave("figures/Fig2_paired_ROC.png", p_roc, width=8.5, height=7, dpi=300, bg="white")
cat("Saved: figures/Fig2_paired_ROC.png\n")
p_roc

In [ ]:
# Paper-ready summary
cat(strrep("=",72),"\n TABLE 1\n",strrep("=",72),"\n")
cat(sprintf("YF n=%d  |  OF n=%d\n",nrow(young_demo),nrow(older_demo)))
table1_var("Age",           "Age (years)",young_demo,older_demo)
table1_var("Screening.BMI","BMI (kg/m2)",young_demo,older_demo)

cat(strrep("=",72),"\n AIM 1\n",strrep("=",72),"\n")
cat(sprintf("%-22s  %-32s  %-32s  %s\n",
            "Marker","YF: AUC (95% CI) [J]","OF: AUC (95% CI) [J]","DeLong p"))
cat(strrep("-",100),"\n")
for (i in seq_along(markers)) {
  m   <- markers[i]; lab <- marker_labels[i]
  ry  <- safe_roc(young$AKI_binary, young[[m]])
  ro  <- safe_roc(older$AKI_binary, older[[m]])
  sy  <- roc_summary(ry); so <- roc_summary(ro)
  dl  <- delong_unpaired(ry, ro)
  fmt <- function(s) {
    if (is.null(s)) return("n/a")
    sprintf("%.2f (%.2f\u2013%.2f) [%.3f]", s$auc, s$ci_lo, s$ci_hi, s$J)
  }
  cat(sprintf("%-22s  %-32s  %-32s  %.4f\n",
              lab, fmt(sy), fmt(so), ifelse(is.na(dl$p), NA, dl$p)))
}

cat("\n",strrep("=",72),"\n AIM 2\n",strrep("=",72),"\n")
cat(sprintf("Spot USG : AUC=%.2f (%.2f-%.2f) | J=%.3f | thr>%.4f | Sens=%.2f | Spec=%.2f\n",
            s_spot$auc,s_spot$ci_lo,s_spot$ci_hi,s_spot$J,s_spot$threshold,s_spot$sens,s_spot$spec))
cat(sprintf("24-hr USG: AUC=%.2f (%.2f-%.2f) | J=%.3f | thr>%.4f | Sens=%.2f | Spec=%.2f\n",
            s_24hr$auc,s_24hr$ci_lo,s_24hr$ci_hi,s_24hr$J,s_24hr$threshold,s_24hr$sens,s_24hr$spec))
cat(sprintf("Paired DeLong: Z=%.3f  p=%.4f\n",dl_aim2$z,dl_aim2$p))